# Interpretability of Video-Informed Humanoid Policies

**Research question.** When we initialize a humanoid control policy from a behavior-cloned reference motion ("video prior"), _what does the network learn that an exploration-only policy does not?_

**Setup.** Two policies trained on the same task, same reward, same architecture:
- **Policy A — from-scratch:** PPO from random init.
- **Policy B — video-prior:** BC pretrain on the retargeted AMASS reference, then PPO.

We then probe the penultimate layer with linear probes, compute layer-wise CKA, and do a top-k unit ablation. See plan Section 3.

**Before running:** execute `python research/scripts/prepare_reference.py --amass research/data/amass_raw/arm_raise.npz --out research/data/reference/arm_raise.npz --fallback`.

## 1. Setup

In [ ]:
from __future__ import annotations

import os
import pathlib
import random

import numpy as np
import torch

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RESEARCH_ROOT = pathlib.Path('..').resolve()
REF_PATH = RESEARCH_ROOT / 'data' / 'reference' / 'arm_raise.npz'
RUNS_DIR = RESEARCH_ROOT / 'runs'
RUNS_DIR.mkdir(exist_ok=True)

assert REF_PATH.exists(), f'Run prepare_reference.py first; expected {REF_PATH}'
ref = np.load(REF_PATH, allow_pickle=True)
QPOS_REF = ref['qpos_ref']           # (T, n_dof)
TARGET_EE = ref['target_ee_pos']     # (3,)
TARGET_QPOS = ref['target_qpos']     # (n_dof,)
META = ref['meta'].item()
print('reference frames:', QPOS_REF.shape, 'target_ee:', TARGET_EE, 'meta:', META)

## 2. Environment definition

Minimal Gymnasium wrapper around the G1 MuJoCo model. Observation = joint pos/vel + end-effector-to-goal vector. Action = delta joint-position targets. Reward = $-\|ee - \text{goal}\|_2 + \alpha \cdot \text{ref\_tracking}$, where the reference-tracking term is only used during Policy B's optional curriculum.

*Note:* this uses a fixed-base arm-raise task (no locomotion) to keep laptop training tractable. See plan risks.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import mujoco

G1_MODEL_CANDIDATES = [
    RESEARCH_ROOT.parent / 'external' / 'unitree_mujoco' / 'unitree_robots' / 'g1' / 'g1.xml',
    RESEARCH_ROOT.parent / 'external' / 'unitree_mujoco' / 'unitree_robots' / 'g1' / 'scene.xml',
]
MODEL_XML = next((p for p in G1_MODEL_CANDIDATES if p.exists()), None)
assert MODEL_XML is not None, f'No G1 MuJoCo XML at any of {G1_MODEL_CANDIDATES}'
print('using model:', MODEL_XML)

In [ ]:
class G1ArmRaiseEnv(gym.Env):
    """Fixed-base arm-raise task. Goal = reach target_ee_pos with right hand."""

    metadata = {'render_modes': []}

    def __init__(self, use_ref_tracking: bool = False, alpha: float = 0.5, max_steps: int = 200):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_path(str(MODEL_XML))
        self.data = mujoco.MjData(self.model)
        self.n_dof = len(META['g1_joint_order'])
        self.use_ref_tracking = use_ref_tracking
        self.alpha = alpha
        self.max_steps = max_steps
        self._step = 0
        self._site_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_SITE, 'right_hand')
        obs_dim = 2 * self.n_dof + 3
        self.observation_space = spaces.Box(-np.inf, np.inf, (obs_dim,), np.float32)
        self.action_space = spaces.Box(-0.1, 0.1, (self.n_dof,), np.float32)
        self.goal = TARGET_EE.astype(np.float32)

    def _qpos_slice(self):
        base = 7 if self.model.nq >= 7 + self.n_dof else 0
        return slice(base, base + self.n_dof)

    def _obs(self):
        s = self._qpos_slice()
        q = np.asarray(self.data.qpos[s], dtype=np.float32)
        v = np.asarray(self.data.qvel[s], dtype=np.float32) if self.model.nv >= s.stop else np.zeros_like(q)
        ee = np.asarray(self.data.site_xpos[self._site_id], dtype=np.float32) if self._site_id >= 0 else np.zeros(3, np.float32)
        return np.concatenate([q, v, ee - self.goal]).astype(np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self._step = 0
        return self._obs(), {}

    def step(self, action):
        s = self._qpos_slice()
        self.data.qpos[s] = np.clip(self.data.qpos[s] + action, -3.14, 3.14)
        mujoco.mj_forward(self.model, self.data)
        self._step += 1
        ee = np.asarray(self.data.site_xpos[self._site_id], dtype=np.float32) if self._site_id >= 0 else np.zeros(3, np.float32)
        dist = float(np.linalg.norm(ee - self.goal))
        reward = -dist
        if self.use_ref_tracking:
            t = min(self._step, QPOS_REF.shape[0] - 1)
            track = float(np.linalg.norm(self.data.qpos[s] - QPOS_REF[t]))
            reward -= self.alpha * track
        terminated = dist < 0.05
        truncated = self._step >= self.max_steps
        return self._obs(), reward, terminated, truncated, {'ee_dist': dist}


probe_env = G1ArmRaiseEnv()
o, _ = probe_env.reset()
print('obs shape:', o.shape, 'act dim:', probe_env.action_space.shape)

## 3. Policy A — from scratch

PPO from random init. Stop at 70% success or 12h wall-clock.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback

TOTAL_TIMESTEPS_A = 500_000  # tune down for laptop, up for Colab A100
POLICY_KWARGS = dict(net_arch=[128, 128])

env_a = DummyVecEnv([lambda: G1ArmRaiseEnv(use_ref_tracking=False)])
model_a = PPO(
    'MlpPolicy', env_a,
    policy_kwargs=POLICY_KWARGS,
    learning_rate=3e-4,
    n_steps=1024, batch_size=256, n_epochs=10,
    gamma=0.99, seed=SEED, verbose=1,
    tensorboard_log=str(RUNS_DIR / 'tb_policy_a'),
)
ckpt_a = CheckpointCallback(save_freq=50_000, save_path=str(RUNS_DIR / 'policy_a_ckpts'))
model_a.learn(total_timesteps=TOTAL_TIMESTEPS_A, callback=ckpt_a)
model_a.save(RUNS_DIR / 'policy_a.zip')

## 4. Policy B — BC pretrain then PPO

Behavior-clone the reference trajectory into an MLP that matches the PPO actor's architecture, then warm-start PPO with those weights.

In [ ]:
import torch.nn as nn
import torch.optim as optim


def build_bc_dataset():
    env = G1ArmRaiseEnv()
    env.reset()
    obs_list, act_list = [], []
    s = env._qpos_slice()
    for t in range(QPOS_REF.shape[0] - 1):
        env.data.qpos[s] = QPOS_REF[t]
        mujoco.mj_forward(env.model, env.data)
        obs_list.append(env._obs())
        act_list.append(np.clip(QPOS_REF[t + 1] - QPOS_REF[t], -0.1, 0.1))
    return np.stack(obs_list).astype(np.float32), np.stack(act_list).astype(np.float32)


obs_bc, act_bc = build_bc_dataset()
print('BC dataset:', obs_bc.shape, act_bc.shape)


class BCActor(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int, hidden=(128, 128)):
        super().__init__()
        layers: list[nn.Module] = []
        prev = obs_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            prev = h
        layers.append(nn.Linear(prev, act_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


bc = BCActor(obs_bc.shape[1], act_bc.shape[1])
opt = optim.Adam(bc.parameters(), lr=1e-3)
X = torch.from_numpy(obs_bc)
Y = torch.from_numpy(act_bc)
for step in range(5_000):
    idx = torch.randint(0, X.shape[0], (256,))
    pred = bc(X[idx])
    loss = ((pred - Y[idx]) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        print(f'step {step} loss {loss.item():.5f}')

In [ ]:
env_b = DummyVecEnv([lambda: G1ArmRaiseEnv(use_ref_tracking=False)])
model_b = PPO(
    'MlpPolicy', env_b,
    policy_kwargs=POLICY_KWARGS,
    learning_rate=3e-4,
    n_steps=1024, batch_size=256, n_epochs=10,
    gamma=0.99, seed=SEED, verbose=1,
    tensorboard_log=str(RUNS_DIR / 'tb_policy_b'),
)

sb3_layers = [m for m in model_b.policy.mlp_extractor.policy_net.modules() if isinstance(m, nn.Linear)]
bc_layers = [m for m in bc.net.modules() if isinstance(m, nn.Linear)]
for src, dst in zip(bc_layers[:-1], sb3_layers):
    if src.weight.shape == dst.weight.shape:
        dst.weight.data.copy_(src.weight.data)
        dst.bias.data.copy_(src.bias.data)
    else:
        print('skip copy — shape mismatch:', src.weight.shape, 'vs', dst.weight.shape)
if bc_layers[-1].weight.shape == model_b.policy.action_net.weight.shape:
    model_b.policy.action_net.weight.data.copy_(bc_layers[-1].weight.data)
    model_b.policy.action_net.bias.data.copy_(bc_layers[-1].bias.data)

ckpt_b = CheckpointCallback(save_freq=50_000, save_path=str(RUNS_DIR / 'policy_b_ckpts'))
model_b.learn(total_timesteps=TOTAL_TIMESTEPS_A, callback=ckpt_b)
model_b.save(RUNS_DIR / 'policy_b.zip')

## 5. Evaluation — collect activations on shared rollouts

In [ ]:
import h5py

ACT_FILE = RUNS_DIR / 'activations.h5'
N_EVAL_EPISODES = 100


def collect_rollouts(model, label: str, n_eps: int):
    env = G1ArmRaiseEnv()
    obs_all, act_all, info_all = [], [], []
    penult, layer1 = [], []
    hooks = []
    linears = [m for m in model.policy.mlp_extractor.policy_net.modules() if isinstance(m, nn.Linear)]
    tmp = {'h1': None, 'h2': None}
    def h1_hook(_m, _i, o): tmp['h1'] = o.detach().cpu().numpy()
    def h2_hook(_m, _i, o): tmp['h2'] = o.detach().cpu().numpy()
    hooks.append(linears[0].register_forward_hook(h1_hook))
    hooks.append(linears[-1].register_forward_hook(h2_hook))
    for ep in range(n_eps):
        o, _ = env.reset(seed=ep)
        done = False
        while not done:
            a, _ = model.predict(o, deterministic=True)
            obs_all.append(o); act_all.append(a)
            layer1.append(tmp['h1'][0] if tmp['h1'] is not None else np.zeros(128))
            penult.append(tmp['h2'][0] if tmp['h2'] is not None else np.zeros(128))
            o, r, term, trunc, info = env.step(a)
            info_all.append((info.get('ee_dist', np.nan), term, trunc))
            done = term or trunc
    for h in hooks: h.remove()
    with h5py.File(ACT_FILE, 'a') as f:
        g = f.require_group(label)
        for key in ('obs', 'act', 'penult', 'layer1', 'info'):
            if key in g: del g[key]
        g.create_dataset('obs', data=np.stack(obs_all))
        g.create_dataset('act', data=np.stack(act_all))
        g.create_dataset('penult', data=np.stack(penult))
        g.create_dataset('layer1', data=np.stack(layer1))
        g.create_dataset('info', data=np.asarray(info_all, dtype=np.float32))
    return float(np.mean([1.0 if t else 0.0 for (_d, t, _tr) in info_all]))


succ_a = collect_rollouts(model_a, 'policy_a', N_EVAL_EPISODES)
succ_b = collect_rollouts(model_b, 'policy_b', N_EVAL_EPISODES)
print(f'success A: {succ_a:.2%}  B: {succ_b:.2%}')

## 6. Linear probes

Targets:
- **joint targets** (sanity — both policies should score well)
- **ee-distance** (task relevance)
- **COM displacement** (balance feature — expected to diverge)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split


def probe(X, y):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=SEED)
    reg = Ridge(alpha=1.0).fit(X_tr, y_tr)
    return float(reg.score(X_te, y_te)), reg


results = {}
with h5py.File(ACT_FILE, 'r') as f:
    for label in ('policy_a', 'policy_b'):
        g = f[label]
        pen = g['penult'][:]
        obs = g['obs'][:]
        n_dof = (obs.shape[1] - 3) // 2
        targets = {
            'joint_pos': obs[:, :n_dof],
            'ee_dist': np.linalg.norm(obs[:, -3:], axis=1, keepdims=True),
            'com_proxy': obs[:, :n_dof].mean(axis=1, keepdims=True),
        }
        for tgt_name, y in targets.items():
            r2, reg = probe(pen, y)
            results[(label, tgt_name)] = (r2, reg)
            print(f'{label:9s} {tgt_name:10s} R2={r2:.3f}')

## 7. CKA across layers

In [ ]:
def cka(A, B):
    A = A - A.mean(0, keepdims=True)
    B = B - B.mean(0, keepdims=True)
    num = np.linalg.norm(A.T @ B, ord='fro') ** 2
    den = np.linalg.norm(A.T @ A, ord='fro') * np.linalg.norm(B.T @ B, ord='fro')
    return float(num / den) if den > 0 else 0.0


with h5py.File(ACT_FILE, 'r') as f:
    m = np.array([
        [cka(f['policy_a'][la][:], f['policy_b'][lb][:]) for lb in ('layer1', 'penult')]
        for la in ('layer1', 'penult')
    ])
print('CKA matrix (rows=A layers, cols=B layers):')
print(m)

## 8. Top-k unit ablation

Zero the top-10 penultimate units ranked by absolute probe weight (for `com_proxy`), rerun evaluation, measure task-success drop. Compares how load-bearing balance-relevant features are across A and B.

In [ ]:
K = 10
ablation_results = {}
for label, model in (('policy_a', model_a), ('policy_b', model_b)):
    _, reg = results[(label, 'com_proxy')]
    top_units = np.argsort(-np.abs(reg.coef_.flatten()))[:K]
    linears = [m_ for m_ in model.policy.mlp_extractor.policy_net.modules() if isinstance(m_, nn.Linear)]
    last = linears[-1]
    saved = last.weight.data[top_units].clone()
    last.weight.data[top_units] = 0.0
    succ = collect_rollouts(model, f'{label}_ablated', 30)
    last.weight.data[top_units] = saved
    ablation_results[label] = succ
    print(f'{label} success after top-{K} ablation: {succ:.2%}')

## 9. Figures

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Probe R2 bars
tgt_names = ['joint_pos', 'ee_dist', 'com_proxy']
x = np.arange(len(tgt_names))
r2_a = [results[('policy_a', t)][0] for t in tgt_names]
r2_b = [results[('policy_b', t)][0] for t in tgt_names]
axes[0].bar(x - 0.2, r2_a, 0.4, label='A (scratch)')
axes[0].bar(x + 0.2, r2_b, 0.4, label='B (video-prior)')
axes[0].set_xticks(x); axes[0].set_xticklabels(tgt_names)
axes[0].set_ylabel('probe R^2'); axes[0].legend(); axes[0].set_title('Linear probe R^2')

# CKA heatmap
sns.heatmap(m, annot=True, ax=axes[1], xticklabels=['layer1_B','penult_B'], yticklabels=['layer1_A','penult_A'], cmap='viridis')
axes[1].set_title('CKA (A vs B)')

# Ablation
axes[2].bar(['A','B'], [ablation_results['policy_a'], ablation_results['policy_b']])
axes[2].set_ylabel('success rate after top-k ablation')
axes[2].set_title(f'Top-{K} unit ablation')

plt.tight_layout()
plt.savefig(RUNS_DIR / 'interp_figures.png', dpi=150)
plt.show()

## 10. What to write up

- **If B's `com_proxy` R² > A's by a wide margin**, and top-k ablation hurts B more than A: the video prior installs balance-specific features into the hidden representation, not just a better output head. This is the headline.
- **If R²s are close but CKA is low**: the two policies learned functionally similar features in different coordinates. Note the rotation; probe in a rotation-invariant way as follow-up.
- **If R²s are near 1.0 everywhere**: task is too trivial. Re-run with a harder task (e.g., reach-under-perturbation).